# Movie KV pregen (ReasonDB text server bundle, Qwen2.5-0.5B)

Runs [`run_movie_kv_pregen.py`](run_movie_kv_pregen.py) which calls **`KvTextQaModelWrapper.prepare_caches`** from [`kv_cache_text_qa_server_new.py`](kv_cache_text_qa_server_new.py).

**Not used for this task:** `run_cache_generation.py`, `cache_generation_manager.py`, `kv_cache_image_qa_server.py` (image / HF datasets only).

## Kaggle Dataset zip layout (`movie_kv_bundle/`)

Upload a zip containing (paths relative to bundle root):

- `run_movie_kv_pregen.py`
- `kv_cache_text_qa_server_new.py`
- `text_kvpress_patch.py`
- `reasondb/` (stub package)
- `queries_workloads/_workload.yaml`

## Workflow

1. **GPU** + **Internet** + **HF_TOKEN** secret.
2. **Add Data**: `reviews_1000.csv` + the code zip above.
3. Set **`RUN_PRESS`** to one alias per session: `ea` | `kvzip` | `finch_no_cpt` | `finch_with_cpt` (four sessions for all methods).
4. Smoke (`SMOKE_MODE=True`, no checkpoint) then full run (`SMOKE_MODE=False`).
5. Download **Output** before the session ends.

Output layout: `{CACHE_DIR}/{model}/{press_name}/comp{ratio}/cache_entry_{sha256}.pt`

In [ ]:
!pip install -q "kvpress>=0.2" "transformers>=4.45" accelerate pandas tqdm safetensors sentencepiece protobuf pyyaml flask flask-restful

In [ ]:
from __future__ import annotations

import os
import shutil
import sys
from pathlib import Path

try:
    from kaggle_secrets import UserSecretsClient

    _sec = UserSecretsClient()
    for name in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
        try:
            t = _sec.get_secret(name)
            if t:
                os.environ["HF_TOKEN"] = t
                os.environ["HUGGING_FACE_HUB_TOKEN"] = t
                print("Loaded secret:", name)
                break
        except Exception:
            continue
except Exception as e:
    print("Secrets:", e)

BUNDLE_NAME = "movie_kv_bundle"
WORK_BUNDLE = Path("/kaggle/working") / BUNDLE_NAME
base = Path("/kaggle/working/movie_kv_pregen_qwen05b")
CACHE_DIR = base / "caches"
CHECKPOINT_CSV = base / "movie_kv_pregen_checkpoint.csv"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_CSV.parent.mkdir(parents=True, exist_ok=True)

RUN_PRESS = "ea"  # ea | kvzip | finch_no_cpt | finch_with_cpt
_allowed = {"ea", "kvzip", "finch_no_cpt", "finch_with_cpt"}
if RUN_PRESS not in _allowed:
    raise ValueError(f"RUN_PRESS must be one of {_allowed}")

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
TAIL = 500
RATIOS = ["0.2", "0.4", "0.6", "0.8"]

SMOKE_MODE = False
SMOKE_MAX_ROWS = 2
SMOKE_RATIOS = ["0.2"]

print("RUN_PRESS:", RUN_PRESS)
print("WORK_BUNDLE:", WORK_BUNDLE)
print("CACHE_DIR:", CACHE_DIR.resolve())

In [ ]:
from __future__ import annotations

from pathlib import Path

REQUIRED = [
    "run_movie_kv_pregen.py",
    "kv_cache_text_qa_server_new.py",
    "text_kvpress_patch.py",
    "reasondb/backends/kv_cache_base.py",
    "queries_workloads/_workload.yaml",
]


def find_bundle_source() -> Path:
    for p in Path("/kaggle/input").rglob("run_movie_kv_pregen.py"):
        root = p.parent
        if all((root / r).exists() for r in REQUIRED):
            return root
    raise FileNotFoundError(
        "Add a Dataset zip with movie_kv_bundle/ containing run_movie_kv_pregen.py and dependencies."
    )


def find_reviews_csv() -> Path:
    for p in Path("/kaggle/input").rglob("reviews_1000.csv"):
        return p
    raise FileNotFoundError("Add Data: reviews_1000.csv")


SRC_BUNDLE = find_bundle_source()
if WORK_BUNDLE.exists():
    shutil.rmtree(WORK_BUNDLE)
shutil.copytree(SRC_BUNDLE, WORK_BUNDLE)
sys.path.insert(0, str(WORK_BUNDLE))

CSV_PATH = find_reviews_csv()
SCRIPT = WORK_BUNDLE / "run_movie_kv_pregen.py"

for r in REQUIRED:
    assert (WORK_BUNDLE / r).is_file(), f"Missing {r}"

print("Copied bundle from:", SRC_BUNDLE)
print("CSV:", CSV_PATH)
print("SCRIPT:", SCRIPT)

## Smoke test

Set `SMOKE_MODE=True` in the config cell. **No `--checkpoint-csv`** on smoke runs.

In [ ]:
import importlib.util
import shlex
import subprocess

import torch

if not SMOKE_MODE:
    print("SMOKE_MODE is False - skipping smoke.")
else:
    cmd = [
        sys.executable,
        str(SCRIPT),
        "--csv",
        str(CSV_PATH),
        "--cache-dir",
        str(CACHE_DIR),
        "--model",
        MODEL,
        "--tail",
        str(int(TAIL)),
        "--max-rows",
        str(int(SMOKE_MAX_ROWS)),
        "--press-name",
        RUN_PRESS,
        "--compression-ratio",
        *SMOKE_RATIOS,
        "--cpt-csv",
        str(CSV_PATH),
    ]
    if RUN_PRESS in ("ea", "kvzip"):
        cmd.append("--kvzip-patch")

    print("Smoke:", shlex.join(cmd))
    subprocess.check_call(cmd, cwd=str(WORK_BUNDLE))

    from kv_cache_text_qa_server_new import KvTextQaModelWrapper

    press = {
        "ea": "expected_attention",
        "kvzip": "kvzip",
        "finch_no_cpt": "finch",
        "finch_with_cpt": "finch-cachenotes",
    }[RUN_PRESS]
    tag = SMOKE_RATIOS[0].replace(".", "_")
    out_dir = CACHE_DIR / MODEL / press / f"comp{tag}"
    pts = list(out_dir.glob("cache_entry_*.pt"))
    assert pts, f"No .pt under {out_dir}"
    try:
        obj = torch.load(pts[0], map_location="cpu", weights_only=False)
    except TypeError:
        obj = torch.load(pts[0], map_location="cpu")
    print("Smoke OK:", pts[0].name, type(obj))

## Full run

Set `SMOKE_MODE=False`. Uses checkpoint CSV for `(press_name, compression_ratio)` resume.

In [ ]:
import shlex
import subprocess

if SMOKE_MODE:
    raise RuntimeError("Set SMOKE_MODE=False before the full run.")

cmd = [
    sys.executable,
    str(SCRIPT),
    "--csv",
    str(CSV_PATH),
    "--cache-dir",
    str(CACHE_DIR),
    "--model",
    MODEL,
    "--tail",
    str(int(TAIL)),
    "--press-name",
    RUN_PRESS,
    "--compression-ratio",
    *RATIOS,
    "--cpt-csv",
    str(CSV_PATH),
    "--checkpoint-csv",
    str(CHECKPOINT_CSV),
]
if RUN_PRESS in ("ea", "kvzip"):
    cmd.append("--kvzip-patch")

print("Full:", shlex.join(cmd))
subprocess.check_call(cmd, cwd=str(WORK_BUNDLE))

## Optional: zip Output

In [ ]:
# import shutil
# shutil.make_archive("/kaggle/working/movie_kv_qwen05b", "zip", root_dir=str(base))
# print("Wrote /kaggle/working/movie_kv_qwen05b.zip")